# 03 — Statistical Testing

Test associations between predictors and the 30-day readmission target.

- **Categorical predictors**: chi-square test of independence.
- **Numeric predictors**: Welch's t-test (or Mann–Whitney U if the distribution is heavy-tailed).

Apply a multiple-testing correction (Benjamini–Hochberg) before interpreting p-values.

**Inputs**: `data/processed/cleaned.csv`


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

from src.config import PROCESSED_DIR, TARGET

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'cleaned.csv')
df.shape

In [ ]:
y = df[TARGET]
features = df.drop(columns=[TARGET])
numeric_cols = features.select_dtypes('number').columns.tolist()
categorical_cols = features.select_dtypes(exclude='number').columns.tolist()
print('numeric:', numeric_cols)
print('categorical:', categorical_cols)

In [ ]:
rows = []
for col in categorical_cols:
    contingency = pd.crosstab(df[col], y)
    chi2, p, dof, _ = stats.chi2_contingency(contingency)
    rows.append({'feature': col, 'test': 'chi2', 'stat': chi2, 'dof': dof, 'p_raw': p})
chi_results = pd.DataFrame(rows)
chi_results

In [ ]:
rows = []
for col in numeric_cols:
    a = df.loc[y == 0, col].dropna()
    b = df.loc[y == 1, col].dropna()
    if len(a) < 2 or len(b) < 2:
        continue
    t, p = stats.ttest_ind(a, b, equal_var=False)
    rows.append({
        'feature': col, 'test': 'welch_t',
        'stat': t, 'p_raw': p,
        'mean_neg': a.mean(), 'mean_pos': b.mean(),
    })
num_results = pd.DataFrame(rows)
num_results

In [ ]:
all_results = pd.concat([chi_results, num_results], ignore_index=True)
reject, p_adj, _, _ = multipletests(all_results['p_raw'], method='fdr_bh')
all_results['p_fdr'] = p_adj
all_results['significant_fdr_0_05'] = reject
all_results.sort_values('p_fdr')

## Notes

- Welch's t-test is sensible for unequal variances. For heavily skewed numeric features (BNP, creatinine) consider switching to `mannwhitneyu`.
- Report effect sizes alongside p-values (Cohen's d for numerics, Cramér's V for chi-square) — the proposal expects practical, not just statistical, significance.
- Save the result table to `reports/` for the writeup.
